# Test Suite: VADER Sentiment Analysis Module
This notebook executes the unit tests and interactive validations for `src/sentiment_analysis.py`.
It tests:
- Positive review sentiment scoring
- Negative review sentiment scoring
- Neutral factual review sentiment scoring
- Batch DataFrame sentiment enrichment (`analyze_dataframe`)
- Scikit-learn transformer compatibility (`VADERFeatureExtractor`)


In [1]:
import os
import sys
import unittest
import numpy as np
import pandas as pd

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.sentiment_analysis import VADERSentimentAnalyzer, VADERFeatureExtractor
print("Sentiment Analysis module imported successfully!")


Working Directory: e:\ML\mini project 2
Sentiment Analysis module imported successfully!


## Interactive Demonstration 1: Single Review Sentiment Analysis


In [2]:
analyzer = VADERSentimentAnalyzer()

reviews = [
    "This product is absolutely amazing and wonderfully built!",
    "Worst purchase ever! Terribly defective, arrived broken and useless.",
    "The shipment arrived on Tuesday in a blue cardboard box."
]

for rev in reviews:
    res = analyzer.analyze(rev)
    print(f"Review: '{rev}'")
    print(f"  -> Polarity: {res['sentiment_label']}, Compound: {res['compound']:+.3f}, Pos: {res['pos']:.2f}, Neu: {res['neu']:.2f}, Neg: {res['neg']:.2f}\n")


Review: 'This product is absolutely amazing and wonderfully built!'
  -> Polarity: Positive, Compound: +0.861, Pos: 0.59, Neu: 0.41, Neg: 0.00

Review: 'Worst purchase ever! Terribly defective, arrived broken and useless.'
  -> Polarity: Negative, Compound: -0.950, Pos: 0.00, Neu: 0.19, Neg: 0.81

Review: 'The shipment arrived on Tuesday in a blue cardboard box.'
  -> Polarity: Neutral, Compound: +0.000, Pos: 0.00, Neu: 1.00, Neg: 0.00



## Interactive Demonstration 2: DataFrame Batch Sentiment Scoring


In [3]:
df_sample = pd.DataFrame({
    "review_text": [
        "Great quality and fast delivery!",
        "Total waste of money, completely broken.",
        "Average product, neither good nor bad."
    ]
})
df_enriched = analyzer.analyze_dataframe(df_sample)
display(df_enriched)


,review_text,pos,neg,neu,compound,sentiment_label
0,Great quality and fast delivery!,0.523,0.000,0.477,0.6588,Positive
1,"Total waste of money, completely broken.",0.000,0.608,0.392,-0.7346,Negative
2,"Average product, neither good nor bad.",0.000,0.544,0.456,-0.5824,Negative


## Interactive Demonstration 3: Scikit-learn VADER Feature Transformer


In [4]:
extractor = VADERFeatureExtractor()
matrix = extractor.fit_transform(reviews)
print(f"Feature matrix shape: {matrix.shape}")
print(f"Feature names: {extractor.get_feature_names_out()}")
df_features = pd.DataFrame(matrix, columns=extractor.get_feature_names_out())
display(df_features)


Feature matrix shape: (3, 4)
Feature names: ['sentiment_pos', 'sentiment_neg', 'sentiment_neu', 'sentiment_compound']


,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_compound
0,0.588,0.000,0.412,0.8607
1,0.000,0.808,0.192,-0.9501
2,0.000,0.000,1.000,0.0000


## Formal Unittest Suite Runner


In [5]:
class TestSentimentAnalysis(unittest.TestCase):

    def setUp(self):
        self.analyzer = VADERSentimentAnalyzer()
        self.extractor = VADERFeatureExtractor()

    def test_positive_sentiment(self):
        text = "This product is absolutely amazing and wonderfully built!"
        res = self.analyzer.analyze(text)
        self.assertEqual(res["sentiment_label"], "Positive")
        self.assertGreater(res["compound"], 0.2)
        self.assertGreater(res["pos"], 0.1)

    def test_negative_sentiment(self):
        text = "Worst purchase ever! Terribly defective, arrived broken and useless."
        res = self.analyzer.analyze(text)
        self.assertEqual(res["sentiment_label"], "Negative")
        self.assertLess(res["compound"], -0.2)
        self.assertGreater(res["neg"], 0.1)

    def test_neutral_sentiment(self):
        text = "The shipment arrived on Tuesday in a blue cardboard box."
        res = self.analyzer.analyze(text)
        self.assertEqual(res["sentiment_label"], "Neutral")
        self.assertGreater(res["neu"], 0.7)

    def test_dataframe_analysis(self):
        df = pd.DataFrame({
            "review_text": [
                "Great!",
                "Terrible!",
                "Neutral package."
            ]
        })
        enriched = self.analyzer.analyze_dataframe(df)
        self.assertIn("compound", enriched.columns)
        self.assertIn("sentiment_label", enriched.columns)
        self.assertEqual(len(enriched), 3)

    def test_feature_extractor_transformer(self):
        texts = ["Excellent item!", "Awful item!"]
        mat = self.extractor.fit_transform(texts)
        self.assertIsInstance(mat, np.ndarray)
        self.assertEqual(mat.shape, (2, 4))
        self.assertEqual(len(self.extractor.get_feature_names_out()), 4)

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestSentimentAnalysis)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_dataframe_analysis (__main__.TestSentimentAnalysis.test_dataframe_analysis) ... ok
test_feature_extractor_transformer (__main__.TestSentimentAnalysis.test_feature_extractor_transformer) ... ok
test_negative_sentiment (__main__.TestSentimentAnalysis.test_negative_sentiment) ... ok
test_neutral_sentiment (__main__.TestSentimentAnalysis.test_neutral_sentiment) ... ok
test_positive_sentiment (__main__.TestSentimentAnalysis.test_positive_sentiment) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.048s

OK
